# CMS Medicare Part D Prescriber Analysis
### Large-scale healthcare claims analysis with PySpark, ML, and GenAI

**Project goal:** Analyze prescribing behavior based on the publicly available CMS Medicare
Part D dataset — tens of millions of rows, by provider, drug, and region. This demonstrates:
- Working with a **large, complex dataset** (PySpark/Databricks)
- **Statistical analysis** of prescribing patterns
- An **ML model** for prediction/classification
- A **generative AI layer** that explains findings in natural language

Dataset: [Medicare Part D Prescribers - by Provider and Drug](https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug)
(CMS.gov, publicly available, no credentialing required)

## 1. Data ingestion — directly via the CMS API (no manual download)

**Why this approach instead of a manual CSV download:** The full file is ~4 GB, which is too
large for Databricks Community Edition (UI upload limit is ~2GB, plus limited storage/RAM on
the free cluster). Instead, we pull data directly via the API, in smaller "batches" of 5,000
rows each, and build a Spark DataFrame from them — with no upload step at all.

**Step 1:** Go to the dataset page and click the **"Access API"** button (not "Download"):
https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug

**Step 2:** Copy the `dataset_id` (UUID) shown there — replace DATASET_ID below.

CMS API
   ↓
Python requests
   ↓
raw JSON records
   ↓
Spark DataFrame
   ↓
from here onward: PySpark

## Fetching data and creating df_raw

In [0]:
import requests
import time

# Medicare Part D Prescribers - by Provider and Drug
DATASET_ID = "9552739e-3d05-4c1b-8eff-ecabf391e2e5"

BASE_URL = f"https://data.cms.gov/data-api/v1/dataset/{DATASET_ID}/data"

# API pagination
PAGE_SIZE = 5000
MAX_PAGES = 100       # max 500,000 rows

all_rows = []

for page in range(MAX_PAGES):

    offset = page * PAGE_SIZE

    params = {
        "size": PAGE_SIZE,
        "offset": offset
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    batch = response.json()

    # Stop if API has no more data
    if not batch:
        print(f"Reached end of dataset at page {page}.")
        break

    all_rows.extend(batch)

    if (page + 1) % 10 == 0:
        print(
            f"Pages loaded: {page + 1}/{MAX_PAGES} | "
            f"Rows: {len(all_rows):,}"
        )

    time.sleep(0.2)


print(f"\nTotal rows downloaded: {len(all_rows):,}")


# Create Spark DataFrame directly — no Pandas
df_raw = spark.createDataFrame(all_rows)

print(f"Spark DataFrame created.")
print(f"Rows: {df_raw.count():,}")
print(f"Columns: {len(df_raw.columns)}")

## 2. Raw data inspection and data quality checks

In [0]:
# 1. View dataset schema
df_raw.printSchema()

In [0]:
# View representative columns and their actual values

df_raw.select(
    "Prscrbr_NPI",
    "Prscrbr_State_Abrvtn",
    "Prscrbr_Type",
    "Brnd_Name",
    "Gnrc_Name",
    "Tot_Clms",
    "Tot_Benes",
    "Tot_Day_Suply",
    "Tot_Drug_Cst",
    "GE65_Tot_Clms"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

# Check NULL and empty string values per column
missing_stats = df_raw.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
])

missing_stats.show(vertical=True)

## 3. Data cleaning and schema preparation

In [0]:
from pyspark.sql import functions as F

# Integer-like columns
integer_cols = [
    "Tot_Benes",
    "Tot_Clms",
    "Tot_Day_Suply",
    "GE65_Tot_Benes",
    "GE65_Tot_Clms",
    "GE65_Tot_Day_Suply"
]

# Decimal / continuous numeric columns
double_cols = [
    "Tot_30day_Fills",
    "Tot_Drug_Cst",
    "GE65_Tot_30day_Fills",
    "GE65_Tot_Drug_Cst"
]

df_clean = df_raw

for c in integer_cols:
    df_clean = df_clean.withColumn(
        c,
        F.when(
            F.trim(F.col(c)) == "",
            None
        ).otherwise(F.col(c)).cast("long")
    )

for c in double_cols:
    df_clean = df_clean.withColumn(
        c,
        F.when(
            F.trim(F.col(c)) == "",
            None
        ).otherwise(F.col(c)).cast("double")
    )

In [0]:
df_clean.printSchema()

In [0]:
# 11. Verify that cleaning was applied successfully

df_clean.select(
    "Prscrbr_NPI",
    "Prscrbr_Type",
    "Brnd_Name",
    "Tot_Clms",
    "Tot_Benes",
    "Tot_Day_Suply",
    "Tot_Drug_Cst",
    "GE65_Tot_Clms"
).show(20, truncate=False)

In [0]:
# No explicit caching is used because this notebook runs on
# Databricks Serverless compute, where this persistence operation
# is not supported.

## 4. Exploratory Data Analysis (EDA)

Explore prescribing patterns, drug utilization, costs, provider specialties, and geographic differences using PySpark aggregations.

In [0]:
# 12. Descriptive statistics of the main numeric variables

df_clean.select(
    "Tot_30day_Fills",
    "Tot_Benes",
    "Tot_Clms",
    "Tot_Day_Suply",
    "Tot_Drug_Cst"
).summary().show(truncate=False)

In [0]:
# 13. Number of unique providers, drugs, specialties, and states

df_clean.select(
    F.countDistinct("Prscrbr_NPI").alias("unique_providers"),
    F.countDistinct("Brnd_Name").alias("unique_brand_drugs"),
    F.countDistinct("Gnrc_Name").alias("unique_generic_drugs"),
    F.countDistinct("Prscrbr_Type").alias("unique_specialties"),
    F.countDistinct("Prscrbr_State_Abrvtn").alias("unique_states")
).show()

In [0]:
# 14. Top 20 specialties by total number of claims

specialty_stats = (
    df_clean
    .groupBy("Prscrbr_Type")
    .agg(
        F.countDistinct("Prscrbr_NPI").alias("num_providers"),
        F.sum("Tot_Clms").alias("total_claims"),
        F.sum("Tot_Drug_Cst").alias("total_drug_cost")
    )
    .orderBy(F.desc("total_claims"))
)

specialty_stats.show(20, truncate=False)

In [0]:
# 15. Top 20 drugs by total number of claims

drug_stats = (
    df_clean
    .groupBy("Brnd_Name", "Gnrc_Name")
    .agg(
        F.countDistinct("Prscrbr_NPI").alias("num_providers"),
        F.sum("Tot_Clms").alias("total_claims"),
        F.sum("Tot_Drug_Cst").alias("total_drug_cost")
    )
    .orderBy(F.desc("total_claims"))
)

drug_stats.show(20, truncate=False)

In [0]:
# 16. Analysis by state

state_stats = (
    df_clean
    .groupBy("Prscrbr_State_Abrvtn")
    .agg(
        F.countDistinct("Prscrbr_NPI").alias("num_providers"),
        F.sum("Tot_Clms").alias("total_claims"),
        F.sum("Tot_Drug_Cst").alias("total_drug_cost")
    )
    .withColumn(
        "avg_cost_per_claim",
        F.col("total_drug_cost") / F.col("total_claims")
    )
    .orderBy(F.desc("total_claims"))
)

state_stats.show(20, truncate=False)

In [0]:
# 17. Provider-level aggregation and derived metrics

provider_stats = (
    df_clean
    .groupBy(
        "Prscrbr_NPI",
        "Prscrbr_Type",
        "Prscrbr_State_Abrvtn"
    )
    .agg(
        F.countDistinct("Brnd_Name").alias("num_unique_drugs"),
        F.sum("Tot_Clms").alias("total_claims"),
        F.sum("Tot_Drug_Cst").alias("total_drug_cost"),
        F.sum("Tot_Day_Suply").alias("total_day_supply")
    )
    .withColumn(
        "avg_cost_per_claim",
        F.col("total_drug_cost") / F.col("total_claims")
    )
    .orderBy(F.desc("total_drug_cost"))
)

provider_stats.show(20, truncate=False)

## 5. Spark Execution Plans and Performance

Analyze how Spark executes transformations, identify shuffles, and compare narrow and wide transformations using physical execution plans.

In [0]:
# 18. Inspect the Spark execution plan for the provider-level aggregation

provider_stats.explain("formatted")

In [0]:
# 19. Compare narrow and wide transformations

narrow_example = (
    df_clean
    .select(
        "Prscrbr_NPI",
        "Prscrbr_Type",
        "Tot_Clms",
        "Tot_Drug_Cst"
    )
    .filter(F.col("Tot_Clms") > 50)
    .withColumn(
        "cost_per_claim",
        F.col("Tot_Drug_Cst") / F.col("Tot_Clms")
    )
)

narrow_example.explain("formatted")

In [0]:
# 20. Repartition the DataFrame by prescriber state

df_repartitioned = df_clean.repartition(
    20,
    "Prscrbr_State_Abrvtn"
)

df_repartitioned.explain("formatted")

In [0]:
# 21. Reduce the number of partitions with coalesce

df_coalesced = df_repartitioned.coalesce(5)

df_coalesced.explain("formatted")

## 6. Spark SQL Analysis

Use Spark SQL on the cleaned dataset to analyze prescribing patterns and demonstrate interoperability between the DataFrame API and SQL.

In [0]:
# 22. Create a temporary SQL view from the cleaned Spark DataFrame

df_clean.createOrReplaceTempView("cms_prescriptions")

In [0]:
%sql

-- 23. Analyze prescribing activity by specialty using Spark SQL

SELECT
    Prscrbr_Type,
    COUNT(DISTINCT Prscrbr_NPI) AS num_providers,
    SUM(Tot_Clms) AS total_claims,
    ROUND(SUM(Tot_Drug_Cst), 2) AS total_drug_cost,
    ROUND(SUM(Tot_Drug_Cst) / SUM(Tot_Clms), 2) AS avg_cost_per_claim
FROM cms_prescriptions
GROUP BY Prscrbr_Type
ORDER BY total_claims DESC
LIMIT 20;

## 7. Window Functions

Use Spark window functions to rank providers within each medical specialty without collapsing individual provider-level rows.

In [0]:
# 24. Rank providers by total drug cost within each specialty

from pyspark.sql.window import Window

specialty_window = (
    Window
    .partitionBy("Prscrbr_Type")
    .orderBy(F.desc("total_drug_cost"))
)

provider_ranked = (
    provider_stats
    .withColumn(
        "rank_within_specialty",
        F.row_number().over(specialty_window)
    )
)

provider_ranked.select(
    "Prscrbr_NPI",
    "Prscrbr_Type",
    "Prscrbr_State_Abrvtn",
    "total_claims",
    "total_drug_cost",
    "rank_within_specialty"
).filter(
    F.col("rank_within_specialty") <= 3
).orderBy(
    "Prscrbr_Type",
    "rank_within_specialty"
).show(50, truncate=False)

## 8. Spark Joins

Create a provider-level lookup table and join it back to prescription-level data to demonstrate Spark join operations.

In [0]:
# 25. Create a provider-level lookup table

provider_lookup = (
    df_clean
    .select(
        "Prscrbr_NPI",
        "Prscrbr_First_Name",
        "Prscrbr_Last_Org_Name",
        "Prscrbr_Type",
        "Prscrbr_State_Abrvtn"
    )
    .dropDuplicates(["Prscrbr_NPI"])
)

provider_lookup.show(10, truncate=False)

In [0]:
# 26. Join prescription-level data with provider information

prescriptions_enriched = (
    df_clean
    .select(
        "Prscrbr_NPI",
        "Brnd_Name",
        "Gnrc_Name",
        "Tot_Clms",
        "Tot_Benes",
        "Tot_Drug_Cst"
    )
    .join(
        provider_lookup,
        on="Prscrbr_NPI",
        how="left"
    )
)

prescriptions_enriched.select(
    "Prscrbr_NPI",
    "Prscrbr_First_Name",
    "Prscrbr_Last_Org_Name",
    "Prscrbr_Type",
    "Prscrbr_State_Abrvtn",
    "Brnd_Name",
    "Tot_Clms",
    "Tot_Drug_Cst"
).show(20, truncate=False)

In [0]:
# 27. Inspect the execution plan of the left join

prescriptions_enriched.explain("formatted")

## 9. Feature Engineering for Spark ML

Create provider-level features that will be used in a Spark ML classification pipeline.

In [0]:
# 28. Create provider-level ML features

# Note: avg_cost_per_claim (total_drug_cost / total_claims) was deliberately excluded here,
# since the classification label below is derived from total_drug_cost — including it would
# create a near-circular feature-target relationship and inflate model performance.
# avg_day_supply_per_claim is used instead: a genuine behavioral signal (how long a typical
# prescription lasts) that carries no direct mathematical link to the cost-based label.

provider_ml = (
    provider_stats
    .select(
        "Prscrbr_NPI",
        "Prscrbr_Type",
        "Prscrbr_State_Abrvtn",
        "num_unique_drugs",
        "total_claims",
        "total_drug_cost",
        "total_day_supply"
    )
    .withColumn(
        "avg_day_supply_per_claim",
        F.col("total_day_supply") / F.col("total_claims")
    )
)

provider_ml.show(10, truncate=False)

In [0]:
# 29. Create the classification target: high-cost prescriber

# Providers in the top 10% of total drug cost will be labeled as high-cost prescribers
cost_threshold = provider_ml.approxQuantile(
    "total_drug_cost",
    [0.90],
    0.01
)[0]

provider_ml = provider_ml.withColumn(
    "label",
    F.when(
        F.col("total_drug_cost") >= cost_threshold,
        1.0
    ).otherwise(0.0)
)

print(f"90th percentile drug cost threshold: ${cost_threshold:,.2f}")

provider_ml.groupBy("label").count().orderBy("label").show()

In [0]:
# 30. Prepare numerical features for Spark ML

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "num_unique_drugs",
        "total_claims",
        "total_day_supply",
        "avg_day_supply_per_claim"
    ],
    outputCol="features",
    handleInvalid="keep"
)

In [0]:
# 31. Split the data into training and test sets

train_df, test_df = provider_ml.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

train_df.groupBy("label").count().orderBy("label").show()
test_df.groupBy("label").count().orderBy("label").show()

## 10. Spark ML Classification

Train a Spark ML classification pipeline to identify high-cost prescribers based on provider-level prescribing patterns.

In [0]:
# 32. Build and train a Logistic Regression classification pipeline

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    regParam=0.1
)

pipeline = Pipeline(
    stages=[
        assembler,
        lr
    ]
)

model = pipeline.fit(train_df)

predictions = model.transform(test_df)

predictions.select(
    "Prscrbr_NPI",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

In [0]:
# 33. Evaluate the classification model

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print(f"AUC:       {auc_evaluator.evaluate(predictions):.4f}")
print(f"Accuracy:  {accuracy_evaluator.evaluate(predictions):.4f}")
print(f"Precision: {precision_evaluator.evaluate(predictions):.4f}")
print(f"Recall:    {recall_evaluator.evaluate(predictions):.4f}")
print(f"F1 Score:  {f1_evaluator.evaluate(predictions):.4f}")

In [0]:
# 34. Confusion matrix

confusion_matrix = (
    predictions
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_matrix.show()

In [0]:
# 35. Evaluate performance specifically for high-cost prescribers (positive class)

tp = predictions.filter(
    (F.col("label") == 1) & (F.col("prediction") == 1)
).count()

fp = predictions.filter(
    (F.col("label") == 0) & (F.col("prediction") == 1)
).count()

fn = predictions.filter(
    (F.col("label") == 1) & (F.col("prediction") == 0)
).count()

precision_positive = tp / (tp + fp)
recall_positive = tp / (tp + fn)
f1_positive = (
    2 * precision_positive * recall_positive
    / (precision_positive + recall_positive)
)

print(f"Positive-class precision: {precision_positive:.4f}")
print(f"Positive-class recall:    {recall_positive:.4f}")
print(f"Positive-class F1:        {f1_positive:.4f}")

In [0]:
# 36. Evaluate different classification thresholds

from pyspark.ml.functions import vector_to_array

threshold_results = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:

    threshold_predictions = (
        predictions
        .withColumn(
            "prob_positive",
            vector_to_array("probability")[1]
        )
        .withColumn(
            "prediction_threshold",
            (F.col("prob_positive") >= threshold).cast("double")
        )
    )

    tp = threshold_predictions.filter(
        (F.col("label") == 1) &
        (F.col("prediction_threshold") == 1)
    ).count()

    fp = threshold_predictions.filter(
        (F.col("label") == 0) &
        (F.col("prediction_threshold") == 1)
    ).count()

    fn = threshold_predictions.filter(
        (F.col("label") == 1) &
        (F.col("prediction_threshold") == 0)
    ).count()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    threshold_results.append(
        (threshold, precision, recall, f1)
    )

for threshold, precision, recall, f1 in threshold_results:
    print(
        f"Threshold {threshold:.2f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

### ML Result Interpretation

The feature set uses `avg_day_supply_per_claim` instead of `avg_cost_per_claim` to avoid a
near-circular relationship with the cost-based classification label (see note above). Re-run
cells 28–36 on Databricks to regenerate the classification metrics (AUC, accuracy, precision,
recall, F1, and the threshold sweep) with this corrected feature set, then update this section
with the new numbers and a short interpretation of the precision–recall trade-off observed.

## 11. Generative AI Insights

Use aggregated Spark results as structured context for a generative AI model to produce a concise natural-language interpretation of prescribing patterns.

In [0]:
# 37. Prepare aggregated Spark results for the GenAI layer

top_specialties = (
    specialty_stats
    .select(
        "Prscrbr_Type",
        "num_providers",
        "total_claims",
        "total_drug_cost"
    )
    .limit(10)
    .collect()
)

top_drugs = (
    drug_stats
    .select(
        "Brnd_Name",
        "Gnrc_Name",
        "num_providers",
        "total_claims",
        "total_drug_cost"
    )
    .limit(10)
    .collect()
)

top_states = (
    state_stats
    .select(
        "Prscrbr_State_Abrvtn",
        "num_providers",
        "total_claims",
        "avg_cost_per_claim"
    )
    .limit(10)
    .collect()
)

print("Top specialties:", len(top_specialties))
print("Top drugs:", len(top_drugs))
print("Top states:", len(top_states))

In [0]:
# 38. Build a structured prompt for the GenAI model

def rows_to_text(rows):
    return "\n".join(
        [", ".join(f"{key}={value}" for key, value in row.asDict().items())
         for row in rows]
    )

genai_prompt = f"""
You are a healthcare data analyst analyzing CMS Medicare Part D prescribing data.

The following results were produced using Apache Spark from a dataset of
500,000 provider-drug prescription records.

TOP PRESCRIBER SPECIALTIES:
{rows_to_text(top_specialties)}

TOP PRESCRIBED DRUGS:
{rows_to_text(top_drugs)}

TOP STATES BY PRESCRIBING ACTIVITY:
{rows_to_text(top_states)}

Provide a concise analytical summary that:

1. Identifies the most important prescribing patterns.
2. Highlights notable differences in drug costs and utilization.
3. Identifies interesting geographic or specialty-level patterns.
4. Distinguishes high utilization from high cost.
5. Avoids making causal or clinical claims that are not supported by the data.

Write the response as a short executive summary for a healthcare analytics audience.
"""

print(genai_prompt[:4000])

In [0]:
%sql

-- 39. Check whether Databricks AI functions are available

SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Reply with exactly: GenAI connection successful.'
) AS response;

In [0]:
# 40. Generate and display the GenAI executive summary

prompt_df = spark.createDataFrame(
    [(genai_prompt,)],
    ["prompt"]
)

prompt_df.createOrReplaceTempView("genai_input")

genai_result = spark.sql("""
    SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        prompt
    ) AS executive_summary
    FROM genai_input
""")

# Extract the generated text and print it with proper line breaks
summary = genai_result.first()["executive_summary"]

print(summary)

## 12. Conclusion

This project demonstrated an end-to-end large-scale healthcare data analytics workflow using Databricks and Apache Spark.

The workflow included:

- Ingesting 500,000 CMS Medicare Part D prescription records from the CMS API
- Data cleaning and schema preparation with PySpark
- Exploratory analysis using distributed Spark transformations and aggregations
- Analysis of Spark execution plans, shuffles, partitioning, and narrow vs. wide transformations
- Spark SQL, window functions, and joins
- Provider-level feature engineering and classification with Spark ML
- Evaluation of an imbalanced classification problem using AUC, precision, recall, F1, and classification thresholds
- Generative AI analysis using Databricks `ai_query()` to convert aggregated Spark results into a human-readable executive summary

The project illustrates how Spark can be used not only for distributed data processing, but as part of a complete analytics pipeline combining large-scale data engineering, machine learning, and generative AI.